# 04 · Sensitivity sweeps (the paper's figures)

Using the **real HVAC jobs**, we sweep the four things a reviewer will ask about:

- **(a) flexibility** — savings vs `flex_hours` (how much slack we grant HVAC).
- **(b) capacity** — savings vs a per-hour power cap $M$ (a shared-transformer limit).
- **(c) season** — savings by month (winter heating vs summer cooling).
- **(d) emission factors** — savings vs the `OTH` factor (an explicit modeling choice).

In [1]:
# --- standard setup for every notebook in this paper ------------------------
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")                 # find cals (in ../)
from dotenv import load_dotenv; load_dotenv("../.env")  # loads EIA_API_KEY if present
from cals import *          # FACTORS, carbon_intensity, Job, schedule, fifo_baseline, ...
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import nb_utils as U             # guarded data loaders + figure helper (see notebooks/nb_utils.py)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

In [2]:
ci, source = U.get_carbon_intensity()
mix, _ = U.get_fuel_mix()                      # needed for the factor sweep (d)
print("carbon source:", source)
hvac_jobs, run_hours, hvac_df = U.load_hvac_jobs(flex_hours=6)
fifo0 = fifo_baseline(hvac_jobs, ci)["total_gco2"]
print("HVAC jobs:", len(hvac_jobs))

carbon source: EIA ISO-NE (LIVE API)


HVAC jobs: 8363


## (a) Savings vs flexibility (`flex_hours`)

More slack = more cheap hours reachable = more savings, with diminishing returns once the
window already spans the daily CI trough. We rebuild the jobs at each `flex_hours` (it
changes both the window **and** the job set), then compare fifo vs optimal.

In [ ]:
# Baseline = the OBSERVED run hour, not FIFO. The HVAC window is
# [run - flex, run + 1 + flex), so a FIFO start advances the whole building by
# exactly `flex` hours onto measurably cleaner hours -- the control would then be
# a function of the swept parameter and the curve would partly measure the
# baseline moving, not the schedule improving. See adapters.py:66-69.
flex_vals = list(range(2, 9))
sav_flex = []
for f in flex_vals:
    jobs, rh = nrel_to_jobs(hvac_df, utc_offset_hours=-5, flex_hours=f)
    fb = U.do_nothing_gco2(jobs, rh, ci)
    op = schedule(jobs, ci, baseline_hours=rh)["total_gco2"]
    sav_flex.append(U.savings_pct(fb, op))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(flex_vals, sav_flex, "o-", color="#2a9d8f")
ax.set_xlabel("flex_hours (slack granted each HVAC hour)"); ax.set_ylabel("carbon saved (%)")
fig.tight_layout(); U.savefig(fig, "04a_savings_vs_flex.png"); plt.show()
print("flex sweep:", ", ".join("%d h=%.2f%%" % (f, s) for f, s in zip(flex_vals, sav_flex)))

## (b) Savings vs a shared capacity cap $M$

A cap limits how much load may pile into the cheapest hours (e.g. a shared transformer).

**Honest accounting.** Under a tight cap some jobs cannot be shifted. If we simply *dropped*
them, the surviving jobs would flatter the ratio (survivorship bias -- fewer, easier jobs).
Instead we charge every un-shiftable job at its **metered run hour** (the physical
do-nothing) via `baseline_hours=run_hours`. Savings are then non-negative **by construction**
and fall smoothly to ~0 as $M \to 0$.

In [ ]:
# Cap = k x MEDIAN ACTIVE load, swept (carbon_sim.CAP_SWEEP_K), NOT a fraction of
# the observed peak. The cap is a SHIFTING limit, so what decides whether it binds
# is how many typical blocks stack under it -- k median-sized blocks fit by
# construction, which stays comparable across buildings. The MA heat-pump peaks
# are single January resistance-backup hours sitting 15-25x their own median, so a
# peak-derived cap is very loose here; peak is kept as the loosest swept cap,
# not as a non-binding endpoint (it still shows a 0.46 pp gap and 506 fallbacks).
from carbon_sim import CAP_SWEEP_K, WORKING_CAP_K, median_active_kw

base_run = U.do_nothing_gco2(hvac_jobs, run_hours, ci)
opt = schedule(hvac_jobs, ci, baseline_hours=run_hours)
uncapped_pct = U.savings_pct(base_run, opt["total_gco2"])

med = median_active_kw(hvac_jobs)
peak = max(j.power_kw for j in hvac_jobs)
caps = [(f"{k:g}x med", round(k * med, 6)) for k in CAP_SWEEP_K]
caps.append(("peak", round(peak, 6)))
print(f"median active load = {med:.3f} kW | observed peak = {peak:.3f} kW "
      f"({peak/med:.1f}x median)")

# Both arms under the SAME cap: greedy heuristic vs exact MILP optimum. The gap
# is the price of the simple rule, and it exists only where the cap binds.
rows = []
for label, M in caps:
    g = schedule(hvac_jobs, ci, capacity_kw=M, baseline_hours=run_hours)["total_gco2"]
    o = schedule_optimal(hvac_jobs, ci, capacity_kw=M, baseline_hours=run_hours)["total_gco2"]
    rows.append({"cap": label, "M (kW)": round(M, 3),
                 "greedy %": round(U.savings_pct(base_run, g), 2),
                 "optimal %": round(U.savings_pct(base_run, o), 2),
                 "gap pp": round(U.savings_pct(base_run, o) - U.savings_pct(base_run, g), 2)})
cap_table = pd.DataFrame(rows)
display(cap_table)
print("uncapped ceiling: %.2f %%" % uncapped_pct)

work_cap = round(WORKING_CAP_K * med, 6)
wg = schedule(hvac_jobs, ci, capacity_kw=work_cap, baseline_hours=run_hours)["total_gco2"]
wo = schedule_optimal(hvac_jobs, ci, capacity_kw=work_cap, baseline_hours=run_hours)["total_gco2"]
print("WORKING CAP k=%g (M=%.3f kW): greedy %.2f%% | optimal %.2f%% | gap %.2f pp"
      % (WORKING_CAP_K, work_cap, U.savings_pct(base_run, wg), U.savings_pct(base_run, wo),
         U.savings_pct(base_run, wo) - U.savings_pct(base_run, wg)))


# INLINE PREVIEW ONLY -- THIS CELL NO LONGER WRITES figures/04b_savings_vs_capacity.png.
# That file is owned by scripts/fig_savings_vs_capacity.py, which draws the combined
# single-panel HVAC+batch figure on the shared k = M / median-aggregate axis. The plot
# below is the HVAC arm alone, kept because the cap_table above is computed here; saving
# it would silently revert the paper figure to the HVAC-only version. Guarded the same
# way as fig04b() in scripts/restyle_figures_b.py, which is defined but not called.
# The explicit ticks plus NullFormatter below fix matplotlib's log-scale minor tick
# labels, which otherwise collide into unreadable overlapping text.
import matplotlib.ticker as mticker

xs = cap_table["M (kW)"].to_numpy(dtype=float)
gy = cap_table["greedy %"].to_numpy(dtype=float)
oy = cap_table["optimal %"].to_numpy(dtype=float)
wg_pct = U.savings_pct(base_run, wg)
wo_pct = U.savings_pct(base_run, wo)

fig, ax = plt.subplots(figsize=(6.6, 3.9))
# k <= 1 is DEGENERATE, not data: at cap = 1x median half the blocks exceed the cap on
# their own (that is what a median is), so those cells measure forced fallback rather
# than scheduling -- see carbon_sim.CAP_SWEEP_K_REPORTED. Shade it instead of plotting
# it as though it were comparable to the rest of the sweep.
ax.axvspan(xs.min() * 0.82, med, color="0.85", alpha=0.55, zorder=0,
           label=r"degenerate (cap $\leq$ median)")
ax.fill_between(xs, gy, oy, color="#e76f51", alpha=0.15, zorder=1, label="greedy shortfall")
ax.plot(xs, gy, "s-", color="#264653", lw=1.8, ms=6, zorder=3, label="greedy heuristic")
ax.plot(xs, oy, "o--", color="#e76f51", lw=1.8, ms=6, zorder=3, label="MILP optimum")
ax.axhline(uncapped_pct, ls=":", color="gray", zorder=2,
           label=f"uncapped ceiling ({uncapped_pct:.2f}%)")
ax.axvline(work_cap, ls="--", lw=0.9, color="k", alpha=0.6, zorder=2)
ax.annotate(f"working cap k={WORKING_CAP_K:g}\n({work_cap:.2f} kW)\n"
            f"greedy {wg_pct:.2f}% / opt {wo_pct:.2f}%",
            xy=(work_cap, wo_pct), xytext=(work_cap * 1.55, wo_pct * 0.58), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="k", lw=0.9))
ax.set_xscale("log")
ax.set_xticks(xs)
ax.set_xticklabels([f"{v:.2f}" for v in xs], fontsize=9)
ax.xaxis.set_minor_formatter(mticker.NullFormatter())   # stop overlapping log minor labels
ax.set_xlabel(r"per-hour capacity cap $M$ (kW, log scale; ticks = $k\times$median, then peak)")
ax.set_ylabel("carbon saved (%)")
ax.legend(loc="lower right", fontsize=8.5, framealpha=0.95)
fig.tight_layout()
# U.savefig(fig, "04b_savings_vs_capacity.png")   # intentionally NOT called -- see above
plt.show()
print("plotted greedy%:  " + ", ".join(f"{x:.2f}kW={y:.2f}%" for x, y in zip(xs, gy)))
print("plotted optimal%: " + ", ".join(f"{x:.2f}kW={y:.2f}%" for x, y in zip(xs, oy)))

## (c) Seasonal savings (by month)

Bucketing jobs by the month they ran shows *when* the grid offers the most room to shift
(large CI swings + plenty of flexible HVAC load).

In [ ]:
# OPERATING POINT: capped greedy at the HEADLINE cap -- k=3 x median active load
# over the WHOLE year's job pool (M = 3 * median_active_kw(hvac_jobs)), the same
# working cap as cell (b) and the paper's 7.28% headline. NOT uncapped, and NOT a
# per-month median: the cap is a fixed physical limit, so it must not be
# re-derived from each month's own load or the months stop being comparable.
# Baseline = the OBSERVED run hour (see cell (a)).
cap_month = round(WORKING_CAP_K * median_active_kw(hvac_jobs), 6)
print(f"seasonal operating point: greedy, cap = {WORKING_CAP_K:g}x annual median "
      f"= {cap_month:.4f} kW (uncapped ceiling is reported in cell (b))")

month = np.array([pd.Timestamp(run_hours[j.job_id]).month for j in hvac_jobs])
sav_month, base_month, avoid_month = [], [], []
for m in range(1, 13):
    jm = [j for j, mm in zip(hvac_jobs, month) if mm == m]
    rh_m = {j.job_id: run_hours[j.job_id] for j in jm}
    fb = U.do_nothing_gco2(jm, rh_m, ci)
    op = schedule(jm, ci, capacity_kw=cap_month, baseline_hours=rh_m)["total_gco2"]
    sav_month.append(U.savings_pct(fb, op) if fb else 0.0)
    base_month.append(fb); avoid_month.append(fb - op)
base_month = np.array(base_month); avoid_month = np.array(avoid_month)

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(range(1, 13), sav_month, color="#457b9d")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax.set_xlabel("month"); ax.set_ylabel("carbon saved (%)")
fig.tight_layout(); U.savefig(fig, "04c_savings_by_month.png"); plt.show()
names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
print("by month:", ", ".join("%s=%.2f%%" % (n, s) for n, s in zip(names, sav_month)))
# Season means are ENERGY-WEIGHTED: a season's saving is its aggregate ratio
# (total avoided / total baseline), NOT the mean of its three monthly
# percentages. The unweighted mean over-weights low-load months and runs ~1-2 pp
# high here. The paper's season figures are the energy-weighted ones.
for lbl, idx in (("winter (DJF)", [11, 0, 1]), ("spring (MAM)", [2, 3, 4]),
                 ("summer (JJA)", [5, 6, 7]), ("fall (SON)", [8, 9, 10])):
    ew = 100.0 * avoid_month[idx].sum() / base_month[idx].sum()
    print("  %-13s %.2f %%  (unweighted month-mean %.2f %%)"
          % (lbl, ew, np.mean([sav_month[i] for i in idx])))

## (d) Emission-factor sensitivity (`OTH`)

`FACTORS['OTH'] = 230` is a **modeling choice** (AR5 biomass median as a proxy for EIA's
blended "other" bucket); a defensible range is ~130–420. We re-price the *same* fuel mix
under each value (`U.ci_from_factors`) and re-schedule. If savings barely move, the headline
result is robust to this choice.

In [ ]:
# OPERATING POINT: capped greedy at the HEADLINE cap (k=3 x median active load),
# matching cells (b)/(c) and the paper's 7.28% headline. NOT uncapped -- the OTH
# range quoted in the paper is the greedy saving at this cap.
# Grid runs to 700: factors.py records that CarbonCast treats "Other" as 700 and
# "Biomass" as 230 as SEPARATE categories, so 700 is the documented upper
# alternative for EIA's blended OTH bucket, not an arbitrary endpoint.
# Baseline = the OBSERVED run hour (see cell (a)), re-priced under each OTH value.
cap_oth = round(WORKING_CAP_K * median_active_kw(hvac_jobs), 6)
print(f"OTH operating point: greedy, cap = {WORKING_CAP_K:g}x median = {cap_oth:.4f} kW")

oth_vals = [130, 200, 230, 300, 420, 490, 600, 700]
sav_oth = []
for v in oth_vals:
    f = dict(FACTORS); f["OTH"] = v
    ci_v = U.ci_from_factors(mix, f)
    fb = U.do_nothing_gco2(hvac_jobs, run_hours, ci_v)
    op = schedule(hvac_jobs, ci_v, capacity_kw=cap_oth, baseline_hours=run_hours)["total_gco2"]
    sav_oth.append(U.savings_pct(fb, op))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(oth_vals, sav_oth, "^-", color="#e76f51")
ax.axvline(230, ls="--", color="gray", label="default (230, AR5 biomass)")
ax.axvline(700, ls=":", color="gray", label="CarbonCast 'Other' (700)")
ax.set_xlabel("OTH emission factor (gCO2/kWh)"); ax.set_ylabel("carbon saved (%)")
ax.legend()
fig.tight_layout(); U.savefig(fig, "04d_savings_vs_oth_factor.png"); plt.show()
print("savings range across OTH %d..%d: %.2f .. %.2f %%"
      % (min(oth_vals), max(oth_vals), min(sav_oth), max(sav_oth)))
print("per-value:", ", ".join("%d=%.2f%%" % (v, s) for v, s in zip(oth_vals, sav_oth)))

## Robustness across three Massachusetts buildings

Each row uses a real full-year NREL ResStock heat-pump building, the observed HVAC run hour
as its baseline, and real 2019 ISO-NE carbon. The 95% interval resamples complete days,
keeping every HVAC job from a sampled day together.


In [ ]:
if source != "EIA ISO-NE (LIVE API)":
    raise RuntimeError("Real MA robustness results require the EIA ISO-NE carbon signal.")

ma_files = (
    "bldg274807_MA_year.parquet",
    "bldg286081_MA_year.parquet",
    "bldg486202_MA_year.parquet",
)
rows = []
for seed, fname in enumerate(ma_files, start=42):
    jobs, run_hours_b, _ = U.load_hvac_jobs(fname, flex_hours=6, utc_offset_hours=-5)
    # Report at the WORKING CAP, matching cells (b)/(c)/(d) and the paper's 7.28%
    # headline. The uncapped arm answers a different question and is reported in (b).
    cap_kw = round(WORKING_CAP_K * median_active_kw(jobs), 6)
    baseline = U.do_nothing_gco2(jobs, run_hours_b, ci)
    scheduled = schedule(jobs, ci, capacity_kw=cap_kw, baseline_hours=run_hours_b)
    daily = pd.DataFrame({
        "day": [run_hours_b[job.job_id].normalize() for job in jobs],
        "baseline_gco2": [U.price_at(job, run_hours_b[job.job_id], ci) for job in jobs],
        "scheduled_gco2": [U.price_at(job, scheduled["assignments"][job.job_id], ci) for job in jobs],
    }).groupby("day", as_index=True).sum()
    # Resample unit is the whole DAY, so a day's HVAC jobs stay together and intra-day
    # correlation is preserved. This bounds day-sampling variability WITHIN one
    # building-year of 2019 only -- not building-to-building spread, not year-to-year
    # grid variation, not emission-factor uncertainty.
    ci_lower, ci_upper = U.bootstrap_savings_ci(
        daily["baseline_gco2"], daily["scheduled_gco2"], seed=seed
    )
    rows.append({
        "building": fname.removesuffix("_MA_year.parquet"),
        "jobs": len(jobs),
        "working cap (kW)": round(cap_kw, 3),
        "savings %": round(U.savings_pct(baseline, scheduled["total_gco2"]), 2),
        "95% CI lower": round(ci_lower, 2),
        "95% CI upper": round(ci_upper, 2),
        "days": len(daily),
    })
ma_robustness = pd.DataFrame(rows).set_index("building")
display(ma_robustness)

## (f) Average versus marginal-proxy accounting

The average signal is the generation-weighted ISO-NE carbon intensity used throughout this notebook. The second signal is an hourly oil-or-gas marginal-emissions proxy: 650 gCO2/kWh when oil generation is positive, otherwise 490 gCO2/kWh. It is a sensitivity analysis, not measured marginal-emissions data. Each building uses the same jobs, 6-hour flexibility, observed run-hour baseline, and working cap of 3 times that building's median active load under both signals. Delta is average savings minus marginal-proxy savings, so a positive value means the marginal proxy lowers attainable savings.

The full-year breakdown behind this table -- the flex sweep, the uncapped arm, the month-by-month delta, and the mechanism diagnostics -- is reproducible via `python scripts/avg_vs_marginal.py`.

In [ ]:
# marginal_ci_proxy is an alias for marginal_ci (see nb_utils.py): it NaN-masks
# zero-generation hours exactly as carbon_intensity does, so both arms schedule over
# an IDENTICAL feasible set and the delta is attributable to the accounting basis
# alone. The reindex guard below then catches any hour the average signal has and the
# marginal one does not.
marginal_ci = U.marginal_ci_proxy(mix).reindex(ci.index)
if marginal_ci.isna().any():
    raise RuntimeError("Marginal proxy is missing hours from the average CI signal.")

rows = []
for fname in ma_files:
    jobs, run_hours_b, _ = U.load_hvac_jobs(fname, flex_hours=6, utc_offset_hours=-5)
    cap_kw = round(WORKING_CAP_K * median_active_kw(jobs), 6)
    average_baseline = U.do_nothing_gco2(jobs, run_hours_b, ci)
    average_scheduled = schedule(
        jobs, ci, capacity_kw=cap_kw, baseline_hours=run_hours_b
    )["total_gco2"]
    marginal_baseline = U.do_nothing_gco2(jobs, run_hours_b, marginal_ci)
    marginal_scheduled = schedule(
        jobs, marginal_ci, capacity_kw=cap_kw, baseline_hours=run_hours_b
    )["total_gco2"]
    average_savings = U.savings_pct(average_baseline, average_scheduled)
    marginal_savings = U.savings_pct(marginal_baseline, marginal_scheduled)
    rows.append({
        "building": fname.removesuffix("_MA_year.parquet"),
        "working cap (kW)": round(cap_kw, 3),
        "average savings %": round(average_savings, 2),
        "marginal-proxy savings %": round(marginal_savings, 2),
        "delta (pp)": round(average_savings - marginal_savings, 2),
    })

average_vs_marginal = pd.DataFrame(rows).set_index("building")
display(average_vs_marginal)
# NOTE the sign convention: delta = average - marginal, so POSITIVE means the marginal
# proxy LOWERS the attainable saving. The percentage falls partly because the marginal
# baseline is ~2x the average one (every kWh charged >= 490 instead of a mean of 261);
# in absolute gCO2 the marginal arm avoids MORE carbon at this cap. See section 4.9.

## (e) Real Alibaba AI sensitivity sweep

The official Alibaba GPU v2020 task table records completed jobs but no real scheduling
deadline. We therefore keep only `Terminated` tasks and sweep the two explicit assumptions:
per-GPU power and additional deadline flexibility. The trace timestamps are aligned to the
2019 ISO-NE carbon timeline, so this cell fetches the matching real EIA window and raises
instead of falling back to a demo curve.


In [ ]:
import os

ai_csv = U.DATA / "ai" / "pai_task_table.csv"
if not ai_csv.is_file():
    raise FileNotFoundError(f"Missing official Alibaba task table: {ai_csv}")
if not os.environ.get("EIA_API_KEY", "").strip():
    raise RuntimeError("Set EIA_API_KEY in ../.env; real AI results cannot use a demo CI curve.")

gpu_power_vals = (0.3, 0.4, 0.5)
ai_flex_vals = (0, 2, 6, 12, 24)

batch_size = 10_000
completed_tasks = 0
ai_start = ai_end = None
aggregate_parts = []
for ai_raw in iter_pai_task_table(ai_csv, chunksize=batch_size):
    ai_parsed = parse_alibaba_trace(ai_raw, origin=U.DEMO_ORIGIN)
    if ai_parsed.empty:
        continue
    completed_tasks += len(ai_parsed)
    aggregate_parts.append(aggregate_alibaba_tasks(ai_parsed))
    chunk_start = ai_parsed["submit"].min().floor("h")
    chunk_end = (
        ai_parsed["submit"]
        + pd.to_timedelta(ai_parsed["duration_h"], unit="h")
        + pd.Timedelta(hours=max(ai_flex_vals))
    ).max().ceil("h")
    ai_start = chunk_start if ai_start is None else min(ai_start, chunk_start)
    ai_end = chunk_end if ai_end is None else max(ai_end, chunk_end)
if ai_start is None or ai_end is None:
    raise RuntimeError("The Alibaba table contained no completed GPU tasks.")
ai_groups = (
    pd.concat(aggregate_parts, ignore_index=True)
    .groupby(["earliest_start", "duration_h"], as_index=False)
    .agg(task_count=("task_count", "sum"), total_gpus=("total_gpus", "sum"))
)
ai_mix = fetch_eia_fuel_mix(
    ai_start.date(),
    ai_end.date(),
    "ISO-NE",
    raw_dir=U.DATA / "carbon",
    api_key=os.environ["EIA_API_KEY"],
    use_synthetic_if_missing=False,
)
ai_ci = carbon_intensity(ai_mix)
if ai_ci.empty:
    raise RuntimeError("EIA returned no carbon-intensity values for the Alibaba window.")

rows = []
for gpu_power_kw in gpu_power_vals:
    for flex_hours in ai_flex_vals:
        costs = uncapped_alibaba_costs(
            ai_groups,
            ai_ci,
            gpu_power_kw=gpu_power_kw,
            flex_hours=flex_hours,
        )
        baseline_gco2 = costs["baseline_gco2"]
        scheduled_gco2 = costs["scheduled_gco2"]
        rows.append({
            "source": "Alibaba GPU v2020 + EIA ISO-NE",
            "gpu_power_kw": gpu_power_kw,
            "flex_hours": flex_hours,
            "completed_tasks": completed_tasks,
            "comparable_tasks": costs["comparable_tasks"],
            "unpriced_baseline_tasks": costs["unpriced_baseline_tasks"],
            "baseline_gco2": baseline_gco2,
            "scheduled_gco2": scheduled_gco2,
            "avoided_gco2": baseline_gco2 - scheduled_gco2,
            "savings_pct": U.savings_pct(baseline_gco2, scheduled_gco2),
        })

ai_sweep = pd.DataFrame(rows)
out_dir = U.ROOT / "results"
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "ai_sweep.csv"
ai_sweep.to_csv(out_path, index=False)
print(f"completed tasks: {completed_tasks:,}")
print(f"carbon window: {ai_start} through {ai_end}")
print(f"saved table -> {out_path}")
ai_sweep


In [ ]:
# CURATED FIGURE (restores figures/04e_ai_savings_sweep.png as committed in 98f159a).
# In (e1) the three gpu_power_kw curves are EXACTLY superimposed -- power scales the
# baseline and the schedule by the same factor, so it cancels out of the ratio
# (nb_utils.get_ai_jobs documents this). Plotting three identical lines reads as a
# rendering bug; plot one, assert the invariant, and state it on the axes.
pct = ai_sweep.groupby("flex_hours")["savings_pct"].mean().sort_index()
spread = float(ai_sweep.groupby("flex_hours")["savings_pct"].agg(lambda s: s.max() - s.min()).max())
assert spread < 1e-6, f"savings_pct must not depend on gpu_power_kw (spread {spread:g})"

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(pct.index, pct.to_numpy(), "o-", color="#2a9d8f", lw=1.8, ms=6)
for x, y in zip(pct.index, pct.to_numpy()):
    axes[0].annotate(f"{y:.2f}", (x, y), textcoords="offset points", xytext=(0, 8),
                     ha="center", fontsize=9)
powers = "/".join(f"{p:g}" for p in sorted(ai_sweep["gpu_power_kw"].unique()))
axes[0].text(0.985, 0.05, f"identical for all\ngpu_power_kw ({powers}):\npower cancels in the ratio",
             transform=axes[0].transAxes, ha="right", va="bottom", fontsize=8.5,
             bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.6"))
axes[0].set_ylim(-0.7, float(pct.max()) * 1.20)
axes[0].set_xlabel("assigned flexibility (hours)")
axes[0].set_ylabel("carbon saved (%)")

for gpu_power_kw, group in ai_sweep.groupby("gpu_power_kw", sort=True):
    group = group.sort_values("flex_hours")
    axes[1].plot(group["flex_hours"], group["avoided_gco2"] / 1e6, "o-",
                 label=f"{gpu_power_kw:.1f} kW/GPU")
axes[1].set_xlabel("assigned flexibility (hours)")
axes[1].set_ylabel("avoided emissions (tCO2)")
axes[1].legend(title="assumed power")
fig.tight_layout(); U.savefig(fig, "04e_ai_savings_sweep.png"); plt.show()
print("(e1) plotted: " + ", ".join(f"flex{int(x)}={y:.2f}%" for x, y in pct.items()))